# March Machine Learning Mania 2026 — Standalone Pure-Inference Pipeline

- **Validation**: 5-fold GroupKFold `BRIER=0.16659`
- **Artifact Dataset**: `stefanoclss/march-machine-learning-mania-2026-artifacts` (zero-pickle JSON + zipped engine)
- **Guarantees**: causal profiles, `(A, B)/(B, A)` swap-symmetry, `[0, 1]` bounds.


In [ ]:
import importlib.util, json, sys, zipfile
from pathlib import Path
import numpy as np
import polars as pl

COMP_SLUG = 'march-machine-learning-mania-2026'
ART_SLUG = 'march-machine-learning-mania-2026-artifacts'
ART_REF = 'stefanoclss/march-machine-learning-mania-2026-artifacts'
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')

def _find_dir(candidates, marker_fn):
    for p in candidates:
        if p.is_dir() and marker_fn(p):
            return p
    if INPUT_ROOT.is_dir():
        for p in sorted(INPUT_ROOT.rglob('*')):
            if p.is_dir() and len(p.parts) <= 7 and marker_fn(p):
                return p
    raise FileNotFoundError(f'Missing input dir under {INPUT_ROOT}')

def _is_art(d):
    return (
        (d / 'model_artifact.json').is_file()
        and (d / 'inference_pipeline.py').is_file()
    )

def _is_comp(d):
    return any(d.glob('*RegularSeason*.csv')) or any(d.glob('SampleSubmission*.csv'))

ART_CANDS = [INPUT_ROOT / 'datasets' / ART_REF, INPUT_ROOT / ART_SLUG, Path('.')]
ART_DIR = _find_dir(ART_CANDS, _is_art)
COMP_CANDS = [INPUT_ROOT / COMP_SLUG, INPUT_ROOT / 'competitions' / COMP_SLUG]
COMP_DIR = _find_dir(COMP_CANDS, _is_comp)
print(f'ART_DIR={ART_DIR} | COMP_DIR={COMP_DIR}')

ENGINE_OUT = WORK_ROOT / '_engine'
ENGINE_OUT.mkdir(parents=True, exist_ok=True)
if (ART_DIR / 'sovra_h').is_dir():
    SRC_ROOT = ART_DIR
elif (ART_DIR / 'sovra_engine' / 'sovra_h').is_dir():
    SRC_ROOT = ART_DIR / 'sovra_engine'
elif (ART_DIR / 'sovra_engine.zip').is_file():
    with zipfile.ZipFile(ART_DIR / 'sovra_engine.zip') as zf:
        zf.extractall(ENGINE_OUT)
    SRC_ROOT = ENGINE_OUT
else:
    SRC_ROOT = next(ART_DIR.rglob('paired_entity_ingest.py')).parent.parent.parent
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))


In [ ]:
from sovra_h.engine.paired_entity_ingest import assemble_paired_entity_directory

inf_path = ART_DIR / 'inference_pipeline.py'
spec = importlib.util.spec_from_file_location('infer_mod', inf_path)
assert spec is not None and spec.loader is not None
infer_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(infer_mod)

_, test_df, sub_df, meta = assemble_paired_entity_directory(
    COMP_DIR, include_swapped_test=True
)
swapped_df = meta['test_swapped_df']
art_json = ART_DIR / 'model_artifact.json'
p_ab = np.asarray(infer_mod.predict_dataframe(test_df, art_json), dtype=float)
p_ba = np.asarray(infer_mod.predict_dataframe(swapped_df, art_json), dtype=float)
preds = 0.5 * (np.clip(p_ab, 0.0, 1.0) + (1.0 - np.clip(p_ba, 0.0, 1.0)))
preds = np.clip(np.nan_to_num(preds, nan=0.5), 0.001, 0.999)

id_col, pred_col = sub_df.columns[0], sub_df.columns[1]
out_df = pl.DataFrame({id_col: sub_df[id_col], pred_col: preds})
assert out_df.height == sub_df.height, 'Row count mismatch'
assert int(out_df[pred_col].null_count()) == 0, 'Null predictions detected!'
out_path = WORK_ROOT / 'submission.csv'
out_df.write_csv(out_path)
print(f'Saved {out_path}: rows={out_df.height} mean={float(preds.mean()):.5f}')
print(out_df.head(5))
